<a href="https://colab.research.google.com/github/ankita6876/CHATBOTS---Using-Natural-Language-Processing-and-Tensorflow/blob/main/CHATBOTS_Using_Natural_Language_Processing_and_Tensorflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving intents.json to intents.json


In [ ]:
!pip install nltk

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
import json
import random
import numpy as np
import nltk
from nltk.stem import PorterStemmer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

In [ ]:
with open('intents.json') as file:
    data = json.load(file)

data

{'intents': [{'tag': 'greeting',
   'patterns': ['Hi', 'How are you', 'Is anyone there?', 'Hello', 'Good day'],
   'responses': ['Hello, thanks for visiting',
    'Good to see you again',
    'Hi there, how can I help?'],
   'context_set': ''},
  {'tag': 'goodbye',
   'patterns': ['Bye', 'See you later', 'Goodbye'],
   'responses': ['See you later, thanks for visiting',
    'Have a nice day',
    'Bye! Come back again soon.']},
  {'tag': 'thanks',
   'patterns': ['Thanks', 'Thank you', "That's helpful"],
   'responses': ['Happy to help!', 'Any time!', 'My pleasure']},
  {'tag': 'hours',
   'patterns': ['What hours are you open?',
    'What are your hours?',
    'When are you open?'],
   'responses': ["We're open every day 9am-9pm",
    'Our hours are 9am-9pm every day']},
  {'tag': 'mopeds',
   'patterns': ['Which mopeds do you have?',
    'What kinds of mopeds are there?',
    'What do you rent?'],
   'responses': ['We rent Yamaha, Piaggio and Vespa mopeds',
    'We have Piaggio, Vesp

# Text Preprocessing

In [ ]:
stemmer = PorterStemmer()


In [ ]:
words = []
labels = []
docs_x = []
docs_y = []

for intent in data["intents"]:
    for pattern in intent["patterns"]:
        tokens = nltk.word_tokenize(pattern)
        words.extend(tokens)
        docs_x.append(tokens)
        docs_y.append(intent["tag"])

    if intent["tag"] not in labels:
        labels.append(intent["tag"])

In [ ]:
words = [stemmer.stem(w.lower()) for w in words if w.isalpha()]
words = sorted(list(set(words)))

labels = sorted(labels)

In [ ]:
training = []
output = []

out_empty = [0] * len(labels)

for x, doc in enumerate(docs_x):
    bag = []

    stemmed_words = [stemmer.stem(w.lower()) for w in doc]

    for w in words:
        if w in stemmed_words:
            bag.append(1)
        else:
            bag.append(0)

    output_row = out_empty[:]
    output_row[labels.index(docs_y[x])] = 1

    training.append(bag)
    output.append(output_row)

training = np.array(training)
output = np.array(output)

In [ ]:
model = Sequential()

model.add(Dense(16, input_shape=(len(training[0]),), activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(len(output[0]), activation='softmax'))

model.compile(loss='categorical_crossentropy',
              optimizer=Adam(learning_rate=0.01),
              metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.fit(training, output, epochs=200, verbose=1)

Epoch 1/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.1111 - loss: 2.2088
Epoch 2/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 0.1111 - loss: 2.1471
Epoch 3/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.1852 - loss: 2.0937
Epoch 4/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - accuracy: 0.2963 - loss: 2.0461
Epoch 5/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step - accuracy: 0.3333 - loss: 2.0033
Epoch 6/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step - accuracy: 0.3333 - loss: 1.9606
Epoch 7/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 139ms/step - accuracy: 0.3333 - loss: 1.9162
Epoch 8/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - accuracy: 0.3333 - loss: 1.8689
Epoch 9/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step - accuracy: 0.3704 - loss: 1.8183
Epoch 10/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.3704 - loss: 1.7649
Epoch 11/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - accuracy: 0.3704 - loss: 1.7095
Epoch 12/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 0.4444 - 

In [ ]:
def bag_of_words(sentence, words):
    bag = [0] * len(words)

    sentence_words = nltk.word_tokenize(sentence)
    sentence_words = [stemmer.stem(word.lower()) for word in sentence_words]

    for s in sentence_words:
        for i, w in enumerate(words):
            if w == s:
                bag[i] = 1

    return np.array(bag)

In [ ]:
def chat():
    print("Bot is ready! Type 'quit' to exit.")

    while True:
        inp = input("You: ")
        if inp.lower() == "quit":
            break

        results = model.predict(np.array([bag_of_words(inp, words)]))
        results_index = np.argmax(results)
        tag = labels[results_index]

        for tg in data["intents"]:
            if tg["tag"] == tag:
                responses = tg["responses"]

        print("Bot:", random.choice(responses))

chat()

Bot is ready! Type 'quit' to exit.
You: hello
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
Bot: Hello, thanks for visiting
You: what's your name?
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
Bot: We're open every day 9am-9pm
You: what?
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
Bot: We rent Yamaha, Piaggio and Vespa mopeds
You: no stop
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
Bot: Hi there, how can I help?
You: are you ok?
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
Bot: Hi there, how can I help?
You: fine
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
Bot: Hello, thanks for visiting
You: why?
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
Bot: Hi there, how can I help?
You: quit
